In [ ]:
pip install tqdm

In [68]:
import os
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm
import glob

## 데이터 로드 변경
- train / vaild 데이터 분리

In [81]:
def load_separate_train_val_data(base_path):
    train_texts = []
    val_texts = []

    # 훈련
    tl_folder = 'tl_data'
    folder_path = os.path.join(base_path, tl_folder)
    json_files = glob.glob(os.path.join(folder_path, "*.json"))

    for json_file in json_files:
        try:
            with open(json_file, 'r', encoding='utf-8') as f:
                data = json.load(f)
            qa_text = f"질문: {data['taskinfo']['input']}\n답변: {data['taskinfo']['output']}"
            train_texts.append(qa_text)
        except Exception as e:
            print(f"로딩 실패: {e}")
    
    # 검증
    val_folder = 'vl_data'
    folder_path = os.path.join(base_path, val_folder)
    json_files = glob.glob(os.path.join(folder_path, "*.json"))
    
    for json_file in json_files:
        try:
            with open(json_file, 'r', encoding='utf-8') as f:
                data = json.load(f)
            qa_text = f"질문: {data['taskinfo']['input']}\n답변: {data['taskinfo']['output']}"
            val_texts.append(qa_text)
        except Exception as e:
            print(f"로딩 실패: {e}")
    
    return train_texts, val_texts

In [ ]:
base_path='./'
train_texts, val_texts = load_separate_train_val_data(base_path)

print(f"훈련 데이터: {len(train_texts)}개")
print(f"검증 데이터: {len(val_texts)}개")
print(f"훈련 샘플: {train_texts[0]}")
print(f"검증 샘플: {val_texts[0]}")

## 토크나이저 구성
- 하이퍼파라미터도 미리 설정

In [ ]:
!pip install tokenizers transformers

In [84]:
batch_size = 8
block_size = 64
max_iters = 10000
eval_interval = 300
learning_rate = 1e-4
device = 'cuda' if torch.cuda.is_available() else 'cpu'
eval_iters = 400
n_embed = 64
n_head = 8
n_layer = 4
dropout = 0.3
VOCAB_SIZE = 8000 

In [72]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace
from transformers import PreTrainedTokenizerFast

def create_legal_tokenizer(texts, vocab_size=VOCAB_SIZE):
    """법률 텍스트용 BPE 토크나이저 생성"""
    tokenizer = Tokenizer(BPE(unk_token="<unk>"))
    tokenizer.pre_tokenizer = Whitespace()
    
    trainer = BpeTrainer(
        special_tokens=["<unk>", "<s>", "</s>", "<pad>"],
        vocab_size=vocab_size
    )
    
    def batch_iterator(batch_size=1000):
        for i in range(0, len(texts), batch_size):
            yield texts[i:i + batch_size]
    
    tokenizer.train_from_iterator(batch_iterator(), trainer=trainer)
    
    # Hugging Face 형식으로 변환
    hf_tokenizer = PreTrainedTokenizerFast(
        tokenizer_object=tokenizer,
        unk_token="<unk>",
        bos_token="<s>",
        eos_token="</s>",
        pad_token="<pad>"
    )
    
    return hf_tokenizer

## 데이터 전처리

In [82]:
def preprocess_legal_data(texts, tokenizer, block_size):
    encoded_data = []

    #토크나이징
    for text in texts:
        tokens = tokenizer(text, add_special_tokens=True)['input_ids']
        
        # if len(tokens) >= block_size + 1:
        if len(tokens) >= 32:
            for i in range(0, len(tokens) - block_size, block_size // 2): 
                if i + block_size + 1 <= len(tokens):
                    encoded_data.append(torch.tensor(tokens[i:i + block_size + 1], dtype=torch.long))
    
    print(f"생성된 학습 시퀀스 수: {len(encoded_data)}")
    return encoded_data

In [74]:
def create_dataloader(tensor_data, batch_size, block_size):
    if not tensor_data:
        raise ValueError("tensor_data가 비어있습니다!")
    
    dataset = TensorDataset(
        torch.stack([seq[:block_size] for seq in tensor_data]),
        torch.stack([seq[1:block_size+1] for seq in tensor_data])
    )
    return DataLoader(dataset, batch_size=batch_size, shuffle=True, pin_memory=True)

## GPT 모델 생성

In [75]:
class Head(nn.Module):
    def __init__(self, head_size):
        super().__init__()
        self.key = nn.Linear(n_embed, head_size, bias=False)
        self.query = nn.Linear(n_embed, head_size, bias=False)
        self.value = nn.Linear(n_embed, head_size, bias=False)
        self.register_buffer("tril", torch.tril(torch.ones(block_size, block_size)))
        self.dropout = nn.Dropout(dropout)

    def forward(self, inputs):
        batch_size, sequence_length, embedding_dim = inputs.shape
        keys = self.key(inputs)
        queries = self.query(inputs)
        weights = queries @ keys.transpose(-2, -1) * (embedding_dim ** -0.5)
        weights = weights.masked_fill(self.tril[:sequence_length, :sequence_length] == 0, float("-inf"))
        weights = F.softmax(weights, dim=-1)
        weights = self.dropout(weights)
        values = self.value(inputs)
        output = weights @ values
        return output

class MultiHeadAttention(nn.Module):
    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(n_embed, n_embed)
        self.dropout = nn.Dropout(dropout)

    def forward(self, inputs):
        out = torch.cat([head(inputs) for head in self.heads], dim=-1)
        out = self.dropout(self.proj(out))
        return out

class FeedForward(nn.Module):
    def __init__(self, n_embed):
        super().__init__()
        self.layer = nn.Sequential(
            nn.Linear(n_embed, 4 * n_embed),
            nn.ReLU(),
            nn.Linear(4 * n_embed, n_embed),
            nn.Dropout(dropout),
        )

    def forward(self, input_tensor):
        return self.layer(input_tensor)

class Block(nn.Module):
    def __init__(self, n_embed, n_heads):
        super().__init__()
        head_size = n_embed // n_heads
        self.attention = MultiHeadAttention(n_heads, head_size)
        self.feed_forward = FeedForward(n_embed)
        self.layer_norm1 = nn.LayerNorm(n_embed)
        self.layer_norm2 = nn.LayerNorm(n_embed)

    def forward(self, input_tensor):
        input_tensor = input_tensor + self.attention(self.layer_norm1(input_tensor))
        input_tensor = input_tensor + self.feed_forward(self.layer_norm2(input_tensor))
        return input_tensor

class LawGPT(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding = nn.Embedding(vocab_size, n_embed)
        self.position_embedding = nn.Embedding(block_size, n_embed)
        self.blocks = nn.ModuleList([Block(n_embed, n_head) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(n_embed)
        self.lm_head = nn.Linear(n_embed, vocab_size)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        token_emb = self.token_embedding(idx)
        pos_emb = self.position_embedding(torch.arange(T, device=device))
        x = token_emb + pos_emb
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :]
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

## Loss 평가 함수

In [76]:
@torch.no_grad()
def evaluate(data_loader, model):
    model.eval()
    total_loss = 0
    num_batches = min(eval_iters, len(data_loader))
    
    for i, batch in enumerate(data_loader):
        if i >= num_batches:
            break
        x, y = batch
        x, y = x.to(device), y.to(device)
        logits, loss = model(x, y)
        total_loss += loss.item()
    return total_loss / num_batches

## main 함수

In [77]:
print(f"사용 디바이스: {device}")

사용 디바이스: cuda


In [ ]:
def main():
    # # 데이터 로드 후 토크나이저
    # print("토크나이저 실행 시작")
    # tokenizer = create_legal_tokenizer(texts)
    # vocab_size = len(tokenizer.get_vocab())
    # print(f"어휘 크기: {vocab_size}")
    
    # print("데이터 전처리")
    # tensor_data = preprocess_legal_data(texts, tokenizer, block_size)
    
    # print("train-test 데이터 분리")
    # n = int(0.9 * len(tensor_data))
    # train_data = tensor_data[:n]
    # test_data = tensor_data[n:]
    
    # train_loader = create_dataloader(train_data, batch_size, block_size)
    # test_loader = create_dataloader(test_data, batch_size, block_size)

    # 분리된 데이터로 사용
    # 토크나이저는 전체 데이터로 학습
    print("토크나이저 실행 시작")
    all_texts = train_texts + val_texts
    tokenizer = create_legal_tokenizer(all_texts)

    print("=== QA 텍스트 길이 분석 ===")
    for i, text in enumerate(train_texts[:10]):
        tokens = tokenizer(text, add_special_tokens=False)['input_ids']
        print(f"텍스트 {i}: {len(tokens)} 토큰")
        if i < 3:
            print(f"내용: {text[:100]}...")
    
    # 각각 전처리
    train_data = preprocess_legal_data(train_texts, tokenizer, block_size)
    val_data = preprocess_legal_data(val_texts, tokenizer, block_size)
    
    # 데이터로더 생성 (비율 분할 없음)
    train_loader = create_dataloader(train_data, batch_size, block_size)
    test_loader = create_dataloader(val_data, batch_size, block_size)
    
    print(f"훈련 데이터 시퀀스 수: {len(train_data)}")
    print(f"테스트 데이터 시퀀스 수: {len(val_data)}")

    print("모델 초기화")
    vocab_size = len(tokenizer.get_vocab())
    model = LawGPT(vocab_size).to(device)
    print(f"모델의 파라미터 수: {sum(p.numel() for p in model.parameters())/1e6:.2f}M")
    
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=0.01)
    
    print("훈련 시작")
    step = 0
    train_iter = iter(train_loader)
    
    with tqdm(total=max_iters, desc="Training") as pbar:
        while step < max_iters:
            # 로스 평가
            if step % eval_interval == 0:
                train_loss = evaluate(train_loader, model)
                val_loss = evaluate(test_loader, model)
                print(f'\nstep: {step}, train loss: {train_loss:.4f}, val loss: {val_loss:.4f}')
            
            # 훈련
            model.train()
            try:
                batch = next(train_iter)
            except StopIteration:
                train_iter = iter(train_loader)
                batch = next(train_iter)
            
            x, y = batch
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
            
            logits, loss = model(x, y)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            step += 1
            pbar.update(1)
            pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    print("훈련 완료!")
    test_text_generation(model, tokenizer, device)

if __name__ == "__main__":
    main()

토크나이저 실행 시작



=== QA 텍스트 길이 분석 ===
텍스트 0: 66 토큰
내용: 질문: 동업계약상 손해배상을 청구할 때 어떤 증거가 필요할까요?
답변: 동업계약에서 손해배상을 청구하려면 귀책사유를 입증할 증거가 필요합니다. 이러한 증거는 계약 위반 사실, 손해...
텍스트 1: 69 토큰
내용: 질문: 불가분채무와 연대채무의 차이점은 무엇인가요?
답변: 불가분채무는 채무자가 여러 명일 때 채무의 성질상 나눌 수 없는 채무를 의미하며, 연대채무는 각 채무자가 독립적으로 전체...
텍스트 2: 83 토큰
내용: 질문: 회사가 근로자에게 작업 환경에서 발생한 사고에 대한 손해배상 책임이 있는 경우, 이를 면탈하기 위한 조건은 무엇인가요?
답변: 회사가 작업 환경에서 발생한 사고에 대한 손해...
텍스트 3: 87 토큰
텍스트 4: 91 토큰
텍스트 5: 109 토큰
텍스트 6: 100 토큰
텍스트 7: 98 토큰
텍스트 8: 81 토큰
텍스트 9: 50 토큰
생성된 학습 시퀀스 수: 51721
생성된 학습 시퀀스 수: 6476
훈련 데이터 시퀀스 수: 51721
테스트 데이터 시퀀스 수: 6476
모델 초기화
모델의 파라미터 수: 1.24M
훈련 시작


Training:   0%|          | 0/10000 [00:00<?, ?it/s]


step: 0, train loss: 9.1537, val loss: 9.1515

step: 300, train loss: 7.8319, val loss: 7.8291


## 테스트

In [54]:
def test_text_generation(model, tokenizer, device, test_prompts=None):
    model.eval()
    
    if test_prompts is None:
        test_prompts = [
            "질문: 계약 위반시 손해배상 범위는?\n답변:",
            "질문: 이행거절의 법적 효과는?\n답변:",
            "질문: 부당이득 반환 요건은?\n답변:"
        ]
    
    print("법률 질문 답변 생성 테스트")
    
    for prompt in test_prompts:
        print(f"{prompt}")
        
        try:
            context_encoded = torch.tensor([tokenizer(prompt, add_special_tokens=False)['input_ids']]).to(device)
            generated_ids = model.generate(context_encoded, max_new_tokens=50)[0]
            generated_text = tokenizer.decode(generated_ids.tolist())
            print(f"생성: {generated_text}")
        except Exception as e:
            print(f"생성 실패: {e}")
            
test_text_generation(model, tokenizer, device)

NameError: name 'model' is not defined